# 01 · Limpieza de datos
    **Proyecto final · Predicción de entregas tardías en Olist (e-commerce, Brasil)**


> **Flujo del proyecto** (diagrama del profesor)
> `01 Limpieza` → `02 Exploración` → `03 Creación de variables` → `04 Preparación (split temporal + folds)` →
> `05a Logística` · `05b Random Forest` · `05c XGBoost` · `05d Red neuronal` (cada uno: selección de variables → hiperparámetros) →
> `06 Selección del mejor modelo y prueba final en test`


    **Entrada:** CSV originales de Olist en `datos/` · **Salida:** tablas limpias en `datos/interim/` (parquet).

    Reglas de esta etapa: no se imputa con estadísticas del conjunto completo (eso se hace dentro de cada fold),
    no se eliminan outliers (los pedidos extremos son justamente los que más se retrasan) y se documenta cada fila eliminada.

In [1]:
import sys, json, time, warnings
from pathlib import Path

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from src import config, viz
viz.aplicar_estilo()
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

from src import limpieza as L

In [2]:
raw = L.cargar_tablas()
display(pd.DataFrame({k: {"filas": v.shape[0], "columnas": v.shape[1]} for k, v in raw.items()}).T)

,filas,columnas
customers,99441,5
sellers,3095,4
reviews,99224,7
items,112650,7
products,32951,9
geolocation,1000163,5
orders,99441,8
payments,103886,5


### Customers

In [3]:
c = raw["customers"]
print("customer_id duplicados:", c.duplicated("customer_id").sum())
print("Clientes únicos con varias compras:", c.duplicated("customer_unique_id").sum())
customers = L.limpiar_customers(c)   # código postal a 5 dígitos + estados oficiales de Brasil
print("Eliminados por estado inválido:", len(c) - len(customers))
display(customers["customer_state"].value_counts(normalize=True).mul(100).round(2).head(10).to_frame("% de clientes"))

customer_id duplicados: 0
Clientes únicos con varias compras: 3345
Eliminados por estado inválido: 0


,% de clientes
customer_state,
SP,41.98
RJ,12.92
MG,11.70
RS,5.50
PR,5.07
SC,3.66
BA,3.40
DF,2.15
ES,2.04


### Sellers

In [4]:
s = raw["sellers"]
print("seller_id duplicados:", s.duplicated("seller_id").sum())
sellers = L.limpiar_sellers(s)
print("Eliminados por estado inválido:", len(s) - len(sellers))
display(sellers["seller_state"].value_counts(normalize=True).mul(100).round(2).head(10).to_frame("% de vendedores"))

seller_id duplicados: 0
Eliminados por estado inválido: 0


,% de vendedores
seller_state,
SP,59.74
PR,11.28
MG,7.88
SC,6.14
RJ,5.53
RS,4.17
GO,1.29
DF,0.97
ES,0.74


### Order Reviews
Las reseñas se escriben **después** de la entrega: nunca entran al modelo (sería fuga de información).
Se usan sólo en la exploración para dimensionar el costo de negocio de un retraso.

In [5]:
reviews = L.limpiar_reviews(raw["reviews"])
display(reviews["review_score"].value_counts(normalize=True).sort_index().mul(100).round(2).to_frame("% de reseñas"))

,% de reseñas
review_score,
1,11.51
2,3.18
3,8.24
4,19.29
5,57.78


### Order Items
`shipping_limit_date` (fecha límite para que el vendedor entregue al transportista) **sí se conoce al momento de la compra**.

In [6]:
items = L.limpiar_items(raw["items"])
display(items[["price", "freight_value"]].describe().round(2))
print("Pedidos con algún ítem con flete 0 (envío gratis):", items.loc[items["freight_value"] == 0, "order_id"].nunique())

,price,freight_value
count,112650.00,112650.00
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,6735.00,409.68


Pedidos con algún ítem con flete 0 (envío gratis): 339


### Products
- Peso 0 no es un peso real → faltante.
- Peso y medidas faltantes **no se imputan aquí**: se imputan dentro del `Pipeline` de cada modelo con la mediana del train de cada fold.
- Categoría faltante → `"sin_categoria"` (es una etiqueta, no un valor estimado).

In [7]:
print("Productos con peso 0:", (raw["products"]["product_weight_g"] == 0).sum())
products = L.limpiar_products(raw["products"])
print("\nFaltantes (se imputan después, dentro de cada fold):")
print(products[L.COLS_DIM].isna().sum())

Productos con peso 0: 4

Faltantes (se imputan después, dentro de cada fold):
product_weight_g     6
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64


### Payments

In [8]:
payments = raw["payments"]
display(payments[["payment_installments", "payment_value"]].describe().round(2))
display(payments["payment_type"].value_counts(normalize=True).mul(100).round(2).to_frame("% de pagos"))

,payment_installments,payment_value
count,103886.00,103886.00
mean,2.85,154.10
std,2.69,217.49
min,0.00,0.00
25%,1.00,56.79
50%,1.00,100.00
75%,4.00,171.84
max,24.00,13664.08


,% de pagos
payment_type,
credit_card,73.92
boleto,19.04
voucher,5.56
debit_card,1.47
not_defined,0.00


### Geolocation · código postal a 5 dígitos y coordenadas dentro de Brasil

In [9]:
geolocation = L.limpiar_geolocation(raw["geolocation"])
print("Registros eliminados por estar fuera de Brasil:", len(raw["geolocation"]) - len(geolocation))

Registros eliminados por estar fuera de Brasil: 40


### Orders y variable objetivo

**Definición del label:** `delivery_late = 1` si el pedido llegó después de la fecha estimada.
`order_estimated_delivery_date` siempre viene a las 00:00, así que comparamos **por día**: un pedido entregado
el mismo día prometido cuenta como a tiempo.

**Reglas de limpieza (en orden):**
1. Se quitan pedidos cancelados / no disponibles.
2. Se **excluyen** pedidos sin fecha de entrega (decisión del equipo).
3. **Censura al final del periodo:** se excluyen pedidos cuya fecha estimada es posterior a la última entrega registrada
   (todavía no se podría saber si llegaron tarde).

**¿Cuándo se conoce el label?** `label_conocida_en = min(entrega, fecha estimada + 1 día)`: si llegó a tiempo se sabe al entregarse;
si llegó tarde se sabe en cuanto vence la fecha estimada. Esta columna se usa para las variables históricas (03) y para la purga del split (04).

In [10]:
o = raw["orders"].copy()
for col in L.FECHAS_ORDERS:
    o[col] = pd.to_datetime(o[col])
display(o["order_status"].value_counts(normalize=True).mul(100).round(2).to_frame("% de pedidos"))
est = o["order_estimated_delivery_date"]
print("¿Todas las fechas estimadas son a las 00:00?", bool((est == est.dt.normalize()).all()))

no_cancelados = o[~o["order_status"].isin(["canceled", "unavailable"])]
diag = (no_cancelados.assign(mes=no_cancelados["order_purchase_timestamp"].dt.to_period("M"),
                             sin_entrega=no_cancelados["order_delivered_customer_date"].isna())
        .groupby("mes")["sin_entrega"].agg(pedidos="size", sin_entrega="sum"))
print("\nPedidos (no cancelados) sin fecha de entrega, últimos 8 meses:")
display(diag.tail(8))

,% de pedidos
order_status,
delivered,97.02
shipped,1.11
canceled,0.63
unavailable,0.61
invoiced,0.32
processing,0.30
created,0.01
approved,0.00


¿Todas las fechas estimadas son a las 00:00? True

Pedidos (no cancelados) sin fecha de entrega, últimos 8 meses:


,pedidos,sin_entrega
mes,,
2018-02,6625,70
2018-03,7168,165
2018-04,6919,121
2018-05,6833,84
2018-06,6145,49
2018-07,6233,77
2018-08,6421,70
2018-09,1,1


In [11]:
orders, bitacora = L.limpiar_orders(raw["orders"])
orders = L.definir_objetivo(orders)
display(pd.Series(bitacora, name="valor").to_frame())

entrega, estimada = orders["order_delivered_customer_date"], orders["order_estimated_delivery_date"]
late_hora = (entrega > estimada).astype(int)
print(f"Tardíos comparando fecha y hora: {late_hora.sum():,} ({late_hora.mean():.2%})")
print(f"Tardíos comparando sólo el día:  {orders['delivery_late'].sum():,} ({orders['delivery_late'].mean():.2%})  <- definición usada")
print(f"Entregados el mismo día estimado (a tiempo con la definición por día): {(late_hora != orders['delivery_late']).sum():,}")

,valor
pedidos_originales,99441
cancelados_o_no_disponibles,1234
sin_fecha_de_entrega,1737
censurados_fecha_estimada_posterior,2
ultima_entrega_registrada,2018-10-17 13:22:46
pedidos_limpios,96468


Tardíos comparando fecha y hora: 7,826 (8.11%)
Tardíos comparando sólo el día:  6,534 (6.77%)  <- definición usada
Entregados el mismo día estimado (a tiempo con la definición por día): 1,292


### Validaciones y guardado

In [12]:
assert orders["order_id"].is_unique
assert orders["delivery_late"].isin([0, 1]).all()
assert orders["order_delivered_customer_date"].notna().all()
assert (orders["label_conocida_en"] >= orders["order_purchase_timestamp"]).mean() > 0.999

tablas = {"orders": orders, "customers": customers, "sellers": sellers, "items": items,
          "products": products, "payments": payments, "geolocation": geolocation, "reviews": reviews}
for nombre, tabla in tablas.items():
    tabla.to_parquet(config.DATOS_INTERIM / f"{nombre}.parquet", index=False)
(config.METRICAS / "limpieza.json").write_text(json.dumps(bitacora, indent=2))
print("Tablas limpias guardadas en", config.DATOS_INTERIM)

Tablas limpias guardadas en /Users/fabiolajrz/Documents/proyecto_final_equipo_3/datos/interim
